# CSS45 spectrum measurement

Rudimentary measurement: connect, measure once, plot the spectrum.

The driver speaks the device's USB HID protocol directly and runs on Linux,
macOS and Windows. Without
hardware the notebook falls back to a synthetic demo spectrum and says so
below the plot. See `PROTOCOL.md` for the wire protocol (hardware-verified).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pycss45 import CSS45, Measurement

In [ ]:
css = None
try:
    css = CSS45()  # auto-discovers the CSS45 over USB HID (VID 0x0C49/PID 0x0010)
    print("connected:", css.get_serial_number())
except (OSError, LookupError) as e:
    print(f"no hardware ({e})")

In [ ]:
if css is not None:
    m = css.measure()  # auto integration time; measure_manual(IntegrationTime.MS10) to pin it
else:
    # demo: a warm-white-ish spectrum, no device needed
    wavelengths = np.linspace(380.0, 780.0, 288)
    m = Measurement(
        wavelengths=wavelengths,
        spectrum=0.05 * (np.exp(-0.5 * ((wavelengths - 600) / 90) ** 2)
                         + 0.35 * np.exp(-0.5 * ((wavelengths - 460) / 60) ** 2)),
        integration_time=0.08,
        cct=3820.0,
        photopic=54.3,
        par=0.021,
    )
    print("using DEMO spectrum")
print(f"integration time {m.integration_time:.3f} s | CCT {m.cct:.0f} K | "
      f"photopic {m.photopic:.2f} | PAR {m.par:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(m.wavelengths, m.spectrum)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("spectral value (W m$^{-2}$ nm$^{-1}$)")
ax.set_title("CSS45 spectrum" if css is not None else "CSS45 spectrum (demo, no hardware)")
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
if css is not None:
    css.disconnect()  # or use `with CSS45() as css:` around a session

Units are SI: nm, s, K. `m.spectrum` and `m.wavelengths` are plain numpy
arrays. For better dark performance run `css.measure_dark_offset()` before
measuring (`css.is_offset_invalid()` tells when the stored offset is stale).